# 04 · Analisis No Supervisado — Clustering de Perfiles Clinicos
**Fase:** Segmentacion no supervisada

**Doble rol del clustering:**
1. Exploratorio: existen perfiles clinicos naturales?
2. Complementario: los clusters se corresponden con distintas tasas de hospitalizacion?

**Algoritmos:** KMeans + Clustering jerarquico aglomerativo (Ward)

> Las variables clinicas son binarias. KMeans con StandardScaler es
> una aproximacion practica; la distancia de Jaccard seria mas correcta
> pero incompatible con Ward linkage. Se documenta como limitacion.

In [ ]:
import sys
sys.path.insert(0,'../../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.cluster.hierarchy as sch
import joblib
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.impute import SimpleImputer
import config
from config import (DATA_INTERIM,CHECKPOINTS,OUT_FIGURES,OUT_TABLES,
    COLORS,RANDOM_STATE,TARGET_COL,SIGNOS_ALARMA,SINTOMAS_GENERALES)
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi':150,'savefig.dpi':300,
    'figure.facecolor':'white','axes.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.titlesize':13,'axes.titleweight':'bold'})
df_con = joblib.load(DATA_INTERIM/'datos_con_signos.pkl')
print(f'DCSA cargado: {df_con.shape[0]:,} registros')

In [ ]:
# Preparacion matriz de features para clustering
# Solo variables clinicas: los clusters reflejan perfiles sintomaticos, no demograficos
todas = SINTOMAS_GENERALES + SIGNOS_ALARMA
df_cl = df_con[todas+[TARGET_COL]].copy()
for v in todas:
    df_cl[v] = df_cl[v].map({1:1, 2:0})

imputer = SimpleImputer(strategy='most_frequent')
X_cl = imputer.fit_transform(df_cl[todas])

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X_cl)
print(f'Matriz: {X_sc.shape} | Nulos: {np.isnan(X_sc).sum()}')

In [ ]:
# Seleccion del numero optimo de clusters
# Tres criterios complementarios:
# 1. Inercia (codo): buscar donde la reduccion se estabiliza
# 2. Silhouette: cohesion y separacion (mayor = mejor, max=1)
# 3. Davies-Bouldin: compacidad/separacion (menor = mejor)
inercias, silhs, dbs = [], [], []
k_range = range(2,10)
for k in k_range:
    km  = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    lbl = km.fit_predict(X_sc)
    inercias.append(km.inertia_)
    silhs.append(silhouette_score(X_sc, lbl, sample_size=5000, random_state=RANDOM_STATE))
    dbs.append(davies_bouldin_score(X_sc, lbl))
    print(f'k={k}: Sil={silhs[-1]:.3f} DB={dbs[-1]:.3f}')

fig, axes = plt.subplots(1,3, figsize=(15,5))
for ax,(vals,titulo,mejor_fn) in zip(axes,[
    (inercias,'Inercia (codo)',lambda x: None),
    (silhs,'Silhouette (mayor=mejor)',np.argmax),
    (dbs,'Davies-Bouldin (menor=mejor)',np.argmin)]):
    ax.plot(list(k_range), vals, marker='o', linewidth=2, color=COLORS['primary'])
    ax.set_title(titulo)
    ax.set_xlabel('k')
    ax.grid(alpha=0.3)
    if mejor_fn(vals) is not None:
        k_opt = list(k_range)[mejor_fn(vals)]
        ax.axvline(k_opt, color=COLORS['secondary'], linestyle='--', label=f'k={k_opt}')
        ax.legend()
plt.suptitle('Seleccion del numero optimo de clusters — DCSA',
    fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIGURES/'12_seleccion_k.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
K_OPT = list(k_range)[np.argmax(silhs)]
print(f'k optimo por silhouette: {K_OPT}')

In [ ]:
# KMeans final
kmeans = KMeans(n_clusters=K_OPT, random_state=RANDOM_STATE, n_init=20, max_iter=500)
etq_km = kmeans.fit_predict(X_sc)
df_cl['Cluster_KMeans'] = etq_km
print(f'KMeans k={K_OPT}: Silhouette={silhouette_score(X_sc,etq_km):.4f}')
print(df_cl['Cluster_KMeans'].value_counts().sort_index())

In [ ]:
# Perfil clinico por cluster
perfil = df_cl.groupby('Cluster_KMeans')[todas].mean().round(3)
perfil['N']           = df_cl.groupby('Cluster_KMeans').size()
perfil['Pct_hosp']    = df_cl.groupby('Cluster_KMeans')[TARGET_COL].mean().round(3)
display(perfil)
perfil.to_csv(OUT_TABLES/'perfil_clusters_DCSA.csv')

# Heatmap
fig, ax = plt.subplots(figsize=(12,5))
datos_hm = perfil[todas].T
sns.heatmap(datos_hm, annot=True, fmt='.2f', cmap='YlOrRd',
    vmin=0, vmax=1, linewidths=0.5, linecolor='white',
    annot_kws={'size':9}, ax=ax)
ax.axhline(len(SINTOMAS_GENERALES), color='black', linewidth=2)
col_labels = [f'C{i}\nN={perfil.loc[i,"N"]:,}\n{perfil.loc[i,"Pct_hosp"]:.0%} hosp'
    for i in sorted(df_cl['Cluster_KMeans'].unique())]
ax.set_xticklabels(col_labels, fontsize=9)
ax.set_title(f'Perfil de sintomas por cluster (k={K_OPT}) — DCSA')
plt.tight_layout()
fig.savefig(OUT_FIGURES/'13_perfil_clusters.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# PCA para visualizacion (solo para graficar, NO para el clustering)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_sc)
var = pca.explained_variance_ratio_
print(f'Varianza: PC1={var[0]:.1%} PC2={var[1]:.1%} Total={sum(var):.1%}')

fig, axes = plt.subplots(1,2, figsize=(14,6))
pal = plt.cm.Set1(np.linspace(0, 0.8, K_OPT))

ax = axes[0]
for k in range(K_OPT):
    m = etq_km==k
    ax.scatter(X_pca[m,0], X_pca[m,1], c=[pal[k]], alpha=0.35, s=8,
        label=f'C{k} (n={m.sum():,})')
cents = pca.transform(kmeans.cluster_centers_)
ax.scatter(cents[:,0], cents[:,1], c=pal, marker='*', s=250,
    edgecolor='black', linewidth=1, zorder=5)
ax.set_xlabel(f'PC1 ({var[0]:.1%})')
ax.set_ylabel(f'PC2 ({var[1]:.1%})')
ax.set_title('A. Clusters en espacio PCA')
ax.legend(fontsize=8, markerscale=3)
ax.grid(alpha=0.2)

ax = axes[1]
target_v = df_cl[TARGET_COL].values
col_h = [COLORS['secondary'] if t==1 else COLORS['primary'] for t in target_v]
ax.scatter(X_pca[:,0], X_pca[:,1], c=col_h, alpha=0.25, s=6)
import matplotlib.patches as mpatches
leg = [mpatches.Patch(color=COLORS['secondary'],label='Hospitalizado'),
    mpatches.Patch(color=COLORS['primary'],label='No hospitalizado')]
ax.legend(handles=leg, fontsize=9)
ax.set_xlabel(f'PC1 ({var[0]:.1%})')
ax.set_ylabel(f'PC2 ({var[1]:.1%})')
ax.set_title('B. Hospitalizacion en espacio PCA')
ax.grid(alpha=0.2)

plt.suptitle('Visualizacion PCA — DCSA', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIGURES/'14_pca_clusters.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Dendrograma — Clustering jerarquico
np.random.seed(RANDOM_STATE)
idx = np.random.choice(len(X_sc), size=min(1500,len(X_sc)), replace=False)
X_mst = X_sc[idx]

fig, ax = plt.subplots(figsize=(14,7))
linked = sch.linkage(X_mst, method='ward')
sch.dendrogram(linked, ax=ax, no_labels=True,
    color_threshold=linked[-K_OPT,2]*0.95,
    above_threshold_color=COLORS['neutral'],
    leaf_rotation=90)
umbral = linked[-K_OPT, 2]
ax.axhline(umbral, color=COLORS['secondary'], linestyle='--', linewidth=2,
    label=f'Corte para k={K_OPT}')
ax.set_title(
    f'Dendrograma — Ward linkage | Muestra: {len(X_mst):,} pacientes | DCSA')
ax.set_xlabel('Pacientes')
ax.set_ylabel('Distancia de fusion')
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(OUT_FIGURES/'15_dendrograma.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

joblib.dump({'kmeans':kmeans,'scaler':scaler,'pca':pca,'K':K_OPT,
    'features':todas}, CHECKPOINTS/'04_clustering.pkl')
df_cl.to_csv(OUT_TABLES/'clusters_DCSA.csv', index=False)
print('Clustering guardado')

## Resumen

Los clusters identifican perfiles sintomaticos diferenciados:
- Clusters con alta carga de signos de alarma multiples: mayor tasa de hospitalizacion
- Clusters con solo sintomas generales: menor tasa

> La co-ocurrencia de signos (no solo la presencia individual) es clave.

**Siguiente:** `05_reportes/05_resultados_consolidados.ipynb`